In [ ]:
import os
import re
import logging
import pdfplumber
import pytesseract
from PIL import Image
import concurrent.futures
import time
from tqdm import tqdm
import config
import io
import sys

In [ ]:
# ———— 正则模式 ————
START_PATTERNS = [
    r"^第[一二三四五六七八九十0-9]{1,3}(?:章|节)\s*(?:管理层讨论与分析|经营情况讨论与分析|董事会报告|董事局报告)\s*$",
    r"^\s*[一二三四五六七八九十0-9]+[、，：:]\s*(?:管理层讨论与分析|经营情况讨论与分析|董事会报告|董事局报告)\s*$"
]
END_PATTERNS = [
    r"^第[四五六七八九十0-9]{1,3}(?:章|节)",
    r"^\s*(?:第\s*)?(?:[一二三四五六七八九十0-9]+)\s*[章节]?\s*[：:]*\s*(?:重要事项|公司治理|财务报告|企业管治报告|监事会报告)",
    r"^\s*(?:重要事项|公司治理|财务报告|企业管治报告|监事会报告)\s*$",
]

START_REGEX = re.compile("|".join(START_PATTERNS))
END_REGEX   = re.compile("|".join(END_PATTERNS))

MDA_KEYWORDS = [
    "总体经营情况", "主营业务分析", "财务状况", "资产负债", "现金流量",
    "投资状况", "经营成果", "行业格局", "发展战略", "研发投入",
    "营业收入", "营业利润", "净利润", "毛利率", "业务回顾",
    "市场分析", "风险因素", "未来展望", "可持续发展"
]

# ———— 提取函数 ————
def extract_mda_from_text(text: str) -> str:
    """
    从整篇年报文本中提取 MDA 部分。
    找到首个 START_REGEX，之后找到第一个 END_REGEX，截取两者之间的内容。
    如果截取后不包含任何 MDA_KEYWORDS，则返回空串。
    """
    lines = text.splitlines()
    start_idx = end_idx = None
    
    # 找起始行
    for i, line in enumerate(lines):
        if START_REGEX.match(line.strip()):
            start_idx = i + 1  # 从下一行开始
            break
    if start_idx is None:
        return ""
    
    # 找结束行
    for j in range(start_idx, len(lines)):
        if END_REGEX.match(lines[j].strip()):
            end_idx = j
            break
    if end_idx is None:
        end_idx = len(lines)
    
    # 提取并校验关键词
    segment = "\n".join(lines[start_idx:end_idx]).strip()
    if any(kw in segment for kw in MDA_KEYWORDS):
        return segment
    else:
        return ""

# ———— 批量处理目录 ————
def batch_extract_mda(txt_dir: str, out_dir: str = None) -> dict:
    """
    遍历 txt_dir 下所有 .txt，提取 MDA 并可选保存到 out_dir。
    返回 {filename: mda_text} 字典。
    """
    results = {}
    os.makedirs(out_dir, exist_ok=True) if out_dir else None
    
    for fname in os.listdir(txt_dir):
        if not fname.lower().endswith(".txt"):
            continue
        path = os.path.join(txt_dir, fname)
        with open(path, encoding="utf-8") as f:
            content = f.read()
        mda = extract_mda_from_text(content)
        if mda:
            results[fname] = mda
            if out_dir:
                out_path = os.path.join(out_dir, fname.replace(".txt", "_MDA.txt"))
                with open(out_path, "w", encoding="utf-8") as fw:
                    fw.write(mda)
    return results



In [ ]:
# ———— 使用示例 ————
if __name__ == "__main__":
    src_dir = "C:/Users/Lenovo/Desktop/上市公司文本处理/项目代码/年报文件/2024/txt年报"
    dst_dir = "C:/Users/Lenovo/Desktop/上市公司文本处理/项目代码\年报文件/2024/txt年报/MDA"
    extracted = batch_extract_mda(src_dir, dst_dir)
    print(f"共提取到 {len(extracted)} 份 MDA 文本，已保存到 {dst_dir}")
    # 如果不需要保存，只需：extracted = batch_extract_mda(src_dir)

In [ ]:
def clean_text(text):
    """清理文本中的常见问题"""
    # 移除页码
    text = re.sub(r'\n\s*\d+\s*\n', '\n', text)
    
    # 移除多余空白行
    text = re.sub(r'\n\s*\n\s*\n+', '\n\n', text)
    
    # 移除页眉页脚
    text = re.sub(
    r'^\s*(\d{0,4}[、\.]?)?\s*[（）()《》〈〉·、.0-9a-zA-Z]{0,50}?(?:第[一二三四五六七八九十0-9]{1,3}[章节]?)?\s*(?:管理层讨论与分析|经营情况讨论与分析|董事会报告|董事局报告|年度报告|年年度报告全文|公司年度报告).*?\s*$',
    '', text,
    flags=re.MULTILINE
)
    merged_lines = []
    buffer = ''

    END_PUNCTUATIONS = ('。', '；', '：', '，', '！', '?', '、', '"', '）')
    PARAGRAPH_STARTS = ('一、', '二、', '三、', '四、', '五、', '六、', '七、', '八、', '九、', '十、',
                        '1、', '2、', '3、', '4、', '5、', '6、', '7、', '8、', '9、', '10、')
    # 移除特殊字符
    text = re.sub(r'[□◇▲●■★△▼]', '', text)
    
    # 简单断行处理 - 尝试处理不正确的断行
    lines = text.split('\n')
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if (line.startswith(PARAGRAPH_STARTS) or buffer.endswith(END_PUNCTUATIONS)):
            if buffer:
                merged_lines.append(buffer)
            buffer = line
        else:
            buffer += line
    
    if buffer:
        merged_lines.append(buffer)
        
    return '\n'.join(merged_lines)

In [ ]:
def main():    
    # 创建清洁输出目录
    clean_dir = os.path.join(dst_dir + "_cleaned")
    if not os.path.exists(clean_dir):
        os.makedirs(clean_dir)
    
    # 获取所有文本文件
    txt_files = [f for f in os.listdir(dst_dir) if f.endswith('.txt')]
    print(f"找到 {len(txt_files)} 个文本文件待清理")
    
    # 使用进度条处理文件
    with tqdm(total=len(txt_files), desc="清理文本") as pbar:
        for txt_file in txt_files:
            try:
                input_path = os.path.join(dst_dir, txt_file)
                output_path = os.path.join(clean_dir, txt_file)
                
                # 读取原文本
                with open(input_path, 'r', encoding='utf-8') as f:
                    text = f.read()
                
                # 清理文本
                cleaned_text = clean_text(text)
                
                # 保存清理后的文本
                with open(output_path, 'w', encoding='utf-8') as f:
                    f.write(cleaned_text)
                
            except Exception as e:
                print('清理失败')
            
            pbar.update(1)
    
    print(f"清理完成! 结果保存在 {clean_dir}")

if __name__ == "__main__":
    main()